# LeetCode 1626 — Best Team With No Conflicts bằng Python

Link đề: https://leetcode.com/problems/best-team-with-no-conflicts/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Sorting

Skeleton LeetCode:

```python
class Solution:
    def bestTeamScore(self, scores: List[int], ages: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến điều kiện "không có xung đột giữa **mọi cặp**" thành điều kiện trên **dãy đã sắp xếp**: sau khi sắp theo `(tuổi, điểm)`, đội hợp lệ ⇔ điểm **không giảm**.
2. Nhận ra bài là biến thể **dãy con không giảm có tổng lớn nhất** (Maximum Sum Increasing Subsequence) và viết DP `O(n²)`.
3. Hiểu vai trò của **tie-break** khi sắp xếp (cùng tuổi phải xếp theo điểm tăng dần).
4. Tối ưu lên `O(n log A)` bằng **cây Fenwick (BIT) lưu max tiền tố**.
5. Viết vét cạn tập con làm oracle và test ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Bạn là quản lý một đội bóng rổ, muốn chọn đội có **tổng điểm** lớn nhất (điểm đội = tổng `scores` của các cầu thủ được chọn).

Đội **không được có xung đột**: xung đột xảy ra khi một cầu thủ **trẻ hơn** có điểm **lớn hơn hẳn** (strictly higher) một cầu thủ **lớn tuổi hơn**. Hai cầu thủ **cùng tuổi** thì **không bao giờ** xung đột.

Cho `scores` và `ages` (cầu thủ `i` có điểm `scores[i]`, tuổi `ages[i]`), trả về tổng điểm lớn nhất của một đội hợp lệ.

**Ví dụ 1:** `scores = [1,3,5,10,15], ages = [1,2,3,4,5]` → `34` (chọn tất cả).

**Ví dụ 2:** `scores = [4,5,6,5], ages = [2,1,2,1]` → `16` (chọn 3 người cuối; được chọn nhiều người cùng tuổi).

**Ví dụ 3:** `scores = [1,2,3,5], ages = [8,9,10,1]` → `6` (chọn 3 người đầu).

**Ràng buộc:**
- `1 ≤ n = scores.length = ages.length ≤ 1000`
- `1 ≤ scores[i] ≤ 10⁶`
- `1 ≤ ages[i] ≤ 1000`

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [
    (([1, 3, 5, 10, 15], [1, 2, 3, 4, 5]), 34),
    (([4, 5, 6, 5], [2, 1, 2, 1]), 16),
    (([1, 2, 3, 5], [8, 9, 10, 1]), 6),
]
for (sc, ag), want in examples:
    print(f"scores={sc}, ages={ag} -> kỳ vọng {want}")

scores=[1, 3, 5, 10, 15], ages=[1, 2, 3, 4, 5] -> kỳ vọng 34
scores=[4, 5, 6, 5], ages=[2, 1, 2, 1] -> kỳ vọng 16
scores=[1, 2, 3, 5], ages=[8, 9, 10, 1] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Điều kiện xung đột viết bằng code

Cặp `(i, j)` xung đột ⇔ (`ages[i] < ages[j]` và `scores[i] > scores[j]`) hoặc ngược lại. Đội hợp lệ ⇔ không cặp nào xung đột.

In [2]:
def conflict(i, j, scores, ages):
    return (ages[i] < ages[j] and scores[i] > scores[j]) or (ages[j] < ages[i] and scores[j] > scores[i])

def valid_team(idx, scores, ages):
    return all(not conflict(i, j, scores, ages) for i, j in combinations(idx, 2))

sc, ag = [4, 5, 6, 5], [2, 1, 2, 1]
print(valid_team([1, 2, 3], sc, ag), valid_team([0, 1], sc, ag))   # [0,1]: người 1 trẻ hơn mà điểm 5 > 4

True False


### 3.2 `zip` + `sorted` với khóa nhiều thành phần

`sorted(zip(ages, scores))` sắp theo tuổi, **tuổi bằng nhau thì theo điểm** (so sánh tuple từ trái sang phải). Đây chính là tie-break ta cần.

In [3]:
players = sorted(zip(ag, sc))
print(players)
print(sorted(zip(ag, sc), key=lambda p: p[0]))          # chỉ theo tuổi: giữ thứ tự gốc khi cùng tuổi (stable)

[(1, 5), (1, 5), (2, 4), (2, 6)]
[(1, 5), (1, 5), (2, 4), (2, 6)]


### 3.3 Bitmask liệt kê tập con

Với `n` nhỏ, mỗi số `mask` trong `0..2ⁿ−1` là một tập con: bit `i` bật ⇔ chọn người `i`.

In [4]:
n = 3
for mask in range(1 << n):
    print(f"{mask:03b} ->", [i for i in range(n) if mask >> i & 1])

000 -> []
001 -> [0]
010 -> [1]
011 -> [0, 1]
100 -> [2]
101 -> [0, 2]
110 -> [1, 2]
111 -> [0, 1, 2]


### 3.4 DP "dãy con tăng có tổng lớn nhất"

Cho dãy `v`, `f[i]` = tổng lớn nhất của dãy con không giảm **kết thúc tại `i`**: `f[i] = v[i] + max(f[j] : j < i, v[j] ≤ v[i])` (max của tập rỗng = 0). Đáp án = `max(f)`.

In [5]:
def max_sum_nondecreasing(v):
    f = [0] * len(v)
    for i in range(len(v)):
        f[i] = v[i] + max([f[j] for j in range(i) if v[j] <= v[i]], default=0)
    return max(f), f

print(max_sum_nondecreasing([5, 5, 4, 6]))

(16, [5, 10, 4, 16])


### 3.5 Cây Fenwick (BIT) cho **max tiền tố**

BIT trên chỉ số `1..A` hỗ trợ: `update(x, val)` — "giá trị tại `x` ít nhất là `val`", `query(x)` — max trên `1..x`. Cả hai `O(log A)`. Lưu ý: BIT max chỉ dùng được khi giá trị **chỉ tăng** (ta không bao giờ giảm một giá trị đã đặt).

In [6]:
class MaxBIT:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
    def update(self, x, val):
        while x <= self.n:
            if self.t[x] < val:
                self.t[x] = val
            x += x & -x
    def query(self, x):
        res = 0
        while x > 0:
            if self.t[x] > res:
                res = self.t[x]
            x -= x & -x
        return res

bit = MaxBIT(10)
bit.update(3, 7); bit.update(6, 4); bit.update(2, 9)
print([bit.query(x) for x in range(1, 11)])

[0, 9, 9, 9, 9, 9, 9, 9, 9, 9]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `scores = [4,5,6,5], ages = [2,1,2,1]`. Sắp theo `(tuổi, điểm)`:

| thứ tự | tuổi | điểm |
|---|---|---|
| 0 | 1 | 5 |
| 1 | 1 | 5 |
| 2 | 2 | 4 |
| 3 | 2 | 6 |

Một đội hợp lệ, khi đọc theo thứ tự này, phải có **điểm không giảm**: nếu một người đứng trước (tuổi nhỏ hơn hoặc bằng) mà điểm lớn hơn người đứng sau thì hoặc là xung đột (tuổi nhỏ hơn hẳn), hoặc… cùng tuổi — nhưng cùng tuổi đã được xếp theo điểm tăng dần nên không thể xảy ra.

Vậy bài toán trở thành: trên dãy điểm `[5, 5, 4, 6]`, tìm **dãy con không giảm có tổng lớn nhất** ⇒ `5 + 5 + 6 = 16` ✔.

Ví dụ 3: `(1,5), (8,1), (9,2), (10,3)` ⇒ dãy điểm `[5, 1, 2, 3]` ⇒ `max(5, 1+2+3) = 6` ✔.

In [7]:
for (sc, ag), want in examples:
    seq = [s for _, s in sorted(zip(ag, sc))]
    best, f = max_sum_nondecreasing(seq)
    print(f"dãy điểm sau sắp xếp {seq} -> f={f} -> {best} (kỳ vọng {want})")

dãy điểm sau sắp xếp [1, 3, 5, 10, 15] -> f=[1, 4, 9, 19, 34] -> 34 (kỳ vọng 34)
dãy điểm sau sắp xếp [5, 5, 4, 6] -> f=[5, 10, 4, 16] -> 16 (kỳ vọng 16)
dãy điểm sau sắp xếp [5, 1, 2, 3] -> f=[5, 1, 3, 6] -> 6 (kỳ vọng 6)


## 5. Cách 1 — Brute Force / baseline: thử mọi tập con

**Ý tưởng:** liệt kê `2ⁿ` tập con, kiểm tra mọi cặp trong tập, giữ tổng lớn nhất. Đây là định nghĩa trực tiếp của đề nên là **oracle** tin cậy.

In [8]:
def best_brute(scores, ages):
    n = len(scores)
    best = 0
    for mask in range(1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if valid_team(idx, scores, ages):
            best = max(best, sum(scores[i] for i in idx))
    return best

for (sc, ag), want in examples:
    print(sc, ag, best_brute(sc, ag), "kỳ vọng", want)

[1, 3, 5, 10, 15] [1, 2, 3, 4, 5] 34 kỳ vọng 34
[4, 5, 6, 5] [2, 1, 2, 1] 16 kỳ vọng 16
[1, 2, 3, 5] [8, 9, 10, 1] 6 kỳ vọng 6


**Độ phức tạp:** Time `O(2ⁿ · n²)`, Space `O(n)`. Với `n = 20` đã là `4·10⁸` — không thể với `n = 1000`.

**Nút thắt:** điều kiện "mọi cặp" buộc ta kiểm tra `O(n²)` cặp cho **mỗi** tập. Nếu sắp xếp trước, điều kiện "mọi cặp" co lại thành "mọi cặp **liền kề**" (tính bắc cầu của `≤`) ⇒ có cấu trúc dãy để làm DP.

In [9]:
rnd = random.Random(0)
for n in (10, 13, 15):
    sc = [rnd.randint(1, 20) for _ in range(n)]; ag = [rnd.randint(1, 6) for _ in range(n)]
    t = time.perf_counter(); best_brute(sc, ag)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=10: 0.001s
n=13: 0.010s
n=15: 0.048s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** sắp các cầu thủ theo `(tuổi tăng, điểm tăng)`. Khi đó:

> Một tập là đội hợp lệ ⇔ dãy điểm của nó (theo thứ tự đã sắp) **không giảm**.

- (⇒) Lấy hai người liên tiếp `p` trước `q` trong đội. Nếu `tuổi_p < tuổi_q` thì không xung đột nghĩa là `điểm_p ≤ điểm_q`. Nếu `tuổi_p = tuổi_q` thì tie-break đã đặt `điểm_p ≤ điểm_q`.
- (⇐) Nếu dãy điểm không giảm thì với hai người bất kỳ `p` trước `q`: `tuổi_p ≤ tuổi_q` và `điểm_p ≤ điểm_q` ⇒ người trẻ hơn (nếu có) không có điểm cao hơn ⇒ không xung đột.

Vậy đáp án = **tổng lớn nhất của dãy con không giảm** trên dãy điểm đã sắp ⇒ DP `O(n²)`.

**Đối xứng:** cũng có thể sắp theo `(điểm tăng, tuổi tăng)` và tìm dãy con có **tuổi** không giảm. Cách sắp này hợp với BIT vì tuổi chỉ tới `1000` (không cần nén tọa độ).

Quy tắc "nếu… thì…":
- **Nếu** điều kiện ràng buộc giữa mọi cặp theo hai thuộc tính **thì** sắp theo một thuộc tính, bài còn lại thành LIS/LNDS trên thuộc tính kia.
- **Nếu** khi sắp có phần tử bằng nhau ở thuộc tính thứ nhất **thì** tie-break theo thuộc tính thứ hai **cùng chiều** (để các phần tử bằng nhau ghép được với nhau).
- **Nếu** DP có dạng `max f[j]` trên `j` có khóa `≤ khóa_i` **thì** dùng BIT/segment tree max tiền tố.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp theo `(tuổi, điểm)` + DP `O(n²)`

Từng đoạn:
1. `players = sorted(zip(ages, scores))`.
2. `f[i] = score_i + max(f[j] với j < i và score_j ≤ score_i)`.
3. Trả `max(f)` (đội tốt nhất có thể kết thúc ở bất kỳ ai).

In [10]:
def best_dp_age(scores, ages):
    players = sorted(zip(ages, scores))
    n = len(players)
    f = [0] * n
    for i in range(n):
        s_i = players[i][1]
        best_prev = 0
        for j in range(i):
            if players[j][1] <= s_i and f[j] > best_prev:
                best_prev = f[j]
        f[i] = best_prev + s_i
    return max(f)

for (sc, ag), want in examples:
    print(best_dp_age(sc, ag), "kỳ vọng", want)

34 kỳ vọng 34
16 kỳ vọng 16
6 kỳ vọng 6


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def bestTeamScore(self, scores: List[int], ages: List[int]) -> int:
        players = sorted(zip(ages, scores))
        f = [0] * len(players)
        for i, (_, s) in enumerate(players):
            best_prev = 0
            for j in range(i):
                if players[j][1] <= s and f[j] > best_prev:
                    best_prev = f[j]
            f[i] = best_prev + s
        return max(f)
```

**Độ phức tạp:** Time `O(n log n + n²)` ≈ `5·10⁵` phép so sánh khi `n = 1000`, Space `O(n)`.

### 7.2 Cách 3 — Sắp theo `(điểm, tuổi)` + DP `O(n²)` trên tuổi

Đối xứng với Cách 2: sau khi sắp theo điểm, chỉ cần dãy **tuổi** không giảm. Viết thêm để thấy hai cách sắp đều đúng (và là bước chuẩn bị cho BIT).

In [11]:
def best_dp_score(scores, ages):
    players = sorted(zip(scores, ages))
    n = len(players)
    f = [0] * n
    for i, (s, a) in enumerate(players):
        best_prev = 0
        for j in range(i):
            if players[j][1] <= a and f[j] > best_prev:
                best_prev = f[j]
        f[i] = best_prev + s
    return max(f)

for (sc, ag), want in examples:
    print(best_dp_score(sc, ag), "kỳ vọng", want)

34 kỳ vọng 34
16 kỳ vọng 16
6 kỳ vọng 6


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def bestTeamScore(self, scores: List[int], ages: List[int]) -> int:
        players = sorted(zip(scores, ages))
        f = [0] * len(players)
        for i, (s, a) in enumerate(players):
            f[i] = s + max((f[j] for j in range(i) if players[j][1] <= a), default=0)
        return max(f)
```

**Độ phức tạp:** Time `O(n²)`, Space `O(n)`.

### 7.3 Cách 4 — Sắp theo `(điểm, tuổi)` + BIT max trên tuổi, `O(n log A)`

Khi duyệt người thứ `i` theo thứ tự `(điểm, tuổi)` tăng, mọi người đã duyệt có điểm `≤ điểm_i`. Ta cần `max f[j]` trên những người đã duyệt có **tuổi `≤ tuổi_i`** ⇒ truy vấn max tiền tố theo tuổi.

Từng đoạn:
1. `bit = MaxBIT(max(ages))`.
2. Với mỗi `(s, a)` theo thứ tự: `cur = bit.query(a) + s`; `bit.update(a, cur)`; cập nhật đáp án.

Vì sao người cùng điểm không gây rắc rối? Hai người cùng điểm không bao giờ xung đột; người đến sau có tuổi `≥` (do tie-break) nên truy vấn của họ bao gồm người đến trước — đúng mong muốn.

In [12]:
def best_bit(scores, ages):
    bit = MaxBIT(max(ages))
    ans = 0
    for s, a in sorted(zip(scores, ages)):
        cur = bit.query(a) + s
        bit.update(a, cur)
        ans = max(ans, cur)
    return ans

for (sc, ag), want in examples:
    print(best_bit(sc, ag), "kỳ vọng", want)

34 kỳ vọng 34
16 kỳ vọng 16
6 kỳ vọng 6


Code nộp LeetCode của Cách 4:

```python
class Solution:
    def bestTeamScore(self, scores: List[int], ages: List[int]) -> int:
        A = max(ages)
        tree = [0] * (A + 1)
        ans = 0
        for s, a in sorted(zip(scores, ages)):
            x, best = a, 0
            while x > 0:                       # max trên tuổi 1..a
                best = max(best, tree[x]); x -= x & -x
            cur = best + s
            x = a
            while x <= A:                      # đặt giá trị mới tại tuổi a
                tree[x] = max(tree[x], cur); x += x & -x
            ans = max(ans, cur)
        return ans
```

**Độ phức tạp:** Time `O(n log n + n log A)`, Space `O(A)` với `A = max(ages) ≤ 1000`.

### 7.4 Cách 5 — Sắp theo `(tuổi, điểm)` + BIT trên điểm đã **nén tọa độ**

Điểm tới `10⁶` — vẫn tạo BIT kích thước `10⁶` được, nhưng gọn hơn là **nén**: thay mỗi điểm bằng thứ hạng của nó trong danh sách điểm phân biệt đã sắp (`1..m`, `m ≤ n`). Sau đó làm y như Cách 4 với vai trò tuổi/điểm hoán đổi.

In [13]:
def best_bit_compressed(scores, ages):
    vals = sorted(set(scores))
    rank = {v: i + 1 for i, v in enumerate(vals)}
    bit = MaxBIT(len(vals))
    ans = 0
    for a, s in sorted(zip(ages, scores)):
        cur = bit.query(rank[s]) + s
        bit.update(rank[s], cur)
        ans = max(ans, cur)
    return ans

for (sc, ag), want in examples:
    print(best_bit_compressed(sc, ag), "kỳ vọng", want)
print("nén tọa độ ví dụ:", {v: i + 1 for i, v in enumerate(sorted(set([1000000, 7, 7, 42])))})

34 kỳ vọng 34
16 kỳ vọng 16
6 kỳ vọng 6
nén tọa độ ví dụ: {7: 1, 42: 2, 1000000: 3}


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`. Đây là kỹ thuật tổng quát khi khóa thứ hai có miền giá trị lớn.

## 8. TRACE từng bước

Trace **Cách 2** trên ví dụ 2 (`players` sau sắp: `(1,5), (1,5), (2,4), (2,6)`):

| i | (tuổi, điểm) | các j hợp lệ (`điểm_j ≤ điểm_i`) | max f[j] | f[i] |
|---|---|---|---|---|
| 0 | (1, 5) | — | 0 | 5 |
| 1 | (1, 5) | 0 | 5 | 10 |
| 2 | (2, 4) | — | 0 | 4 |
| 3 | (2, 6) | 0, 1, 2 | 10 | 16 |

Đáp án `max(f) = 16`.

In [14]:
def trace_dp(scores, ages):
    players = sorted(zip(ages, scores))
    print("sau sắp xếp (tuổi, điểm):", players)
    f = [0] * len(players)
    for i, (a, s) in enumerate(players):
        ok = [j for j in range(i) if players[j][1] <= s]
        best_prev = max((f[j] for j in ok), default=0)
        f[i] = best_prev + s
        print(f"i={i} {players[i]}: j hợp lệ={ok}, max f[j]={best_prev} -> f[{i}]={f[i]}")
    print("f =", f, "-> đáp án", max(f))
    return max(f)

assert trace_dp([4, 5, 6, 5], [2, 1, 2, 1]) == 16

sau sắp xếp (tuổi, điểm): [(1, 5), (1, 5), (2, 4), (2, 6)]
i=0 (1, 5): j hợp lệ=[], max f[j]=0 -> f[0]=5
i=1 (1, 5): j hợp lệ=[0], max f[j]=5 -> f[1]=10
i=2 (2, 4): j hợp lệ=[], max f[j]=0 -> f[2]=4
i=3 (2, 6): j hợp lệ=[0, 1, 2], max f[j]=10 -> f[3]=16
f = [5, 10, 4, 16] -> đáp án 16


In [15]:
def trace_bit(scores, ages):
    A = max(ages); bit = MaxBIT(A); ans = 0
    for s, a in sorted(zip(scores, ages)):
        q = bit.query(a)
        cur = q + s
        bit.update(a, cur)
        ans = max(ans, cur)
        snapshot = [bit.query(x) for x in range(1, A + 1)]
        print(f"(điểm={s}, tuổi={a}): max tuổi≤{a} = {q} -> cur={cur}; max tiền tố theo tuổi 1..{A} = {snapshot}")
    print("đáp án:", ans)
    return ans

assert trace_bit([1, 2, 3, 5], [8, 9, 10, 1]) == 6

(điểm=1, tuổi=8): max tuổi≤8 = 0 -> cur=1; max tiền tố theo tuổi 1..10 = [0, 0, 0, 0, 0, 0, 0, 1, 1, 1]
(điểm=2, tuổi=9): max tuổi≤9 = 1 -> cur=3; max tiền tố theo tuổi 1..10 = [0, 0, 0, 0, 0, 0, 0, 1, 3, 3]
(điểm=3, tuổi=10): max tuổi≤10 = 3 -> cur=6; max tiền tố theo tuổi 1..10 = [0, 0, 0, 0, 0, 0, 0, 1, 3, 6]
(điểm=5, tuổi=1): max tuổi≤1 = 0 -> cur=5; max tiền tố theo tuổi 1..10 = [5, 5, 5, 5, 5, 5, 5, 5, 5, 6]
đáp án: 6


## 9. Code nộp LeetCode

Chọn Cách 2 (dễ hiểu, `n ≤ 1000` nên `O(n²)` thoải mái), hoặc Cách 4 nếu muốn nhanh hơn. Dưới đây là Cách 4.

In [16]:
class Solution:
    def bestTeamScore(self, scores: List[int], ages: List[int]) -> int:
        A = max(ages)
        tree = [0] * (A + 1)
        ans = 0
        for s, a in sorted(zip(scores, ages)):
            x, best = a, 0
            while x > 0:
                if tree[x] > best:
                    best = tree[x]
                x -= x & -x
            cur = best + s
            x = a
            while x <= A:
                if tree[x] < cur:
                    tree[x] = cur
                x += x & -x
            if cur > ans:
                ans = cur
        return ans

for (sc, ag), want in examples:
    got = Solution().bestTeamScore(sc, ag)
    print(got, "OK" if got == want else "SAI")

34 OK
16 OK
6 OK


## 10. Vì sao thuật toán đúng

1. **Rút gọn (mục 6):** đội hợp lệ ⇔ dãy con không giảm (theo khóa thứ hai) của danh sách đã sắp theo `(khóa 1, khóa 2)`. Hai chiều đã chứng minh ở mục 6; điểm then chốt là tie-break cùng chiều.
2. **DP (Cách 2/3):** `f[i]` = tổng lớn nhất của dãy con hợp lệ **kết thúc tại `i`**. Phần tử ngay trước `i` trong dãy con đó là một `j < i` với khóa `≤` khóa `i`; phần còn lại là dãy con tối ưu kết thúc tại `j` (cấu trúc con tối ưu — nếu không tối ưu thì thay bằng cái tốt hơn sẽ được tổng lớn hơn). Nếu `i` đứng đầu thì `f[i] = score_i`. Mọi đội có người cuối nào đó ⇒ đáp án `max f`.
3. **BIT (Cách 4/5):** khi xét `i`, BIT chứa đúng `f[j]` của mọi `j` đã xét (`j < i` trong thứ tự sắp) tại vị trí khóa của `j`; `query(khóa_i)` = `max f[j]` với khóa `≤ khóa_i` — đúng bằng biểu thức DP. Giá trị trong BIT chỉ tăng nên BIT max hợp lệ.

## 11. Phân tích Time / Space Complexity

Gọi `n` = số cầu thủ (≤ 1000), `A = max(ages)` (≤ 1000).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2ⁿ · n²)` | `O(n)` |
| 2. Sắp (tuổi, điểm) + DP | `O(n²)` | `O(n)` |
| 3. Sắp (điểm, tuổi) + DP | `O(n²)` | `O(n)` |
| 4. BIT trên tuổi | `O(n log n + n log A)` | `O(A)` |
| 5. BIT trên điểm nén | `O(n log n)` | `O(n)` |

Không phụ thuộc phân bố dữ liệu: average = worst.

In [17]:
rnd = random.Random(7)
sc = [rnd.randint(1, 10**6) for _ in range(1000)]; ag = [rnd.randint(1, 1000) for _ in range(1000)]
for f in (best_dp_age, best_dp_score, best_bit, best_bit_compressed):
    t = time.perf_counter(); v = f(sc, ag)
    print(f"{f.__name__:20s} n=1000 -> {v} trong {time.perf_counter() - t:.3f}s")

best_dp_age          n=1000 -> 32254276 trong 0.017s
best_dp_score        n=1000 -> 32254276 trong 0.018s
best_bit             n=1000 -> 32254276 trong 0.001s
best_bit_compressed  n=1000 -> 32254276 trong 0.001s


## 12. Bộ test

- Ví dụ đề + các ví dụ tự dựng (đã kiểm bằng vét cạn — đối chiếu lại đề gốc).
- Edge: 1 cầu thủ; mọi người cùng tuổi (chọn tất cả); mọi người cùng điểm (chọn tất cả); tuổi tăng nhưng điểm giảm (chỉ chọn được 1 người hoặc nhóm nhỏ).
- Ngẫu nhiên (seed cố định): `n ≤ 12` so với vét cạn; `n ≤ 1000` đúng ràng buộc, các cách nhanh so với nhau.

In [18]:
ALL = [best_dp_age, best_dp_score, best_bit, best_bit_compressed,
       lambda s, a: Solution().bestTeamScore(s, a)]
fixed = examples + [
    (([5], [3]), 5),
    (([5, 4, 6], [1, 1, 2]), 15),
    (([3, 1, 1, 1, 1], [1, 2, 3, 4, 5]), 4),
    (([10, 2, 3, 8], [3, 1, 2, 2]), 23),
    (([6, 5, 4, 3], [1, 2, 3, 4]), 6),
    (([1, 1, 1], [3, 2, 1]), 3),
    (([9, 2, 8, 8, 2], [4, 1, 3, 1, 2]), 27),
    (([7, 1, 9], [5, 5, 5]), 17),
]
for (sc, ag), want in fixed:
    assert best_brute(sc, ag) == want, (sc, ag)
    for f in ALL:
        assert f(sc, ag) == want, (f, sc, ag)
print("ví dụ cố định: OK")

ví dụ cố định: OK


In [19]:
rnd = random.Random(1626)
for _ in range(400):
    n = rnd.randint(1, 11)
    sc = [rnd.randint(1, 8) for _ in range(n)]
    ag = [rnd.randint(1, 5) for _ in range(n)]
    want = best_brute(sc, ag)
    for f in ALL:
        assert f(sc, ag) == want, (f, sc, ag)
for _ in range(10):
    n = rnd.randint(1, 1000)
    sc = [rnd.randint(1, 10**6) for _ in range(n)]
    ag = [rnd.randint(1, 1000) for _ in range(n)]
    vals = {f(sc, ag) for f in ALL}
    assert len(vals) == 1
print("random tests: OK")

random tests: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp theo tuổi nhưng **không** tie-break theo điểm

In [20]:
def wrong_no_tiebreak(scores, ages):
    players = sorted(zip(ages, scores), key=lambda p: p[0])     # SAI: cùng tuổi giữ thứ tự gốc
    f = [0] * len(players)
    for i, (_, s) in enumerate(players):
        f[i] = s + max((f[j] for j in range(i) if players[j][1] <= s), default=0)
    return max(f)

cx = ([5, 4, 6], [1, 1, 2])
print("sai:", wrong_no_tiebreak(*cx), "| đúng:", best_brute(*cx))

sai: 11 | đúng: 15


**Vì sao sai:** hai người cùng tuổi `(1,5)` rồi `(1,4)` — DP coi `5 → 4` là giảm nên không ghép được, dù cùng tuổi **không xung đột**. **Sửa:** `sorted(zip(ages, scores))` (tuple tự tie-break theo điểm tăng dần).

### Lỗi 2 — Dùng `<` thay vì `≤` (dãy tăng **nghiêm ngặt**)

In [21]:
def wrong_strict(scores, ages):
    players = sorted(zip(ages, scores))
    f = [0] * len(players)
    for i, (_, s) in enumerate(players):
        f[i] = s + max((f[j] for j in range(i) if players[j][1] < s), default=0)   # SAI
    return max(f)

cx = ([5, 5], [1, 2])
print("sai:", wrong_strict(*cx), "| đúng:", best_brute(*cx))

sai: 5 | đúng: 10


**Vì sao sai:** xung đột chỉ khi người trẻ có điểm **lớn hơn hẳn**; điểm bằng nhau là hợp lệ. **Sửa:** điều kiện `≤`.

### Lỗi 3 — Tham lam: duyệt theo tuổi, gặp ai không nhỏ hơn người cuối thì lấy

In [22]:
def wrong_greedy(scores, ages):
    total, last = 0, 0
    for a, s in sorted(zip(ages, scores)):
        if s >= last:
            total += s; last = s
    return total

cx = ([3, 1, 1, 1, 1], [1, 2, 3, 4, 5])
print("sai:", wrong_greedy(*cx), "| đúng:", best_brute(*cx))

sai: 3 | đúng: 4


**Vì sao sai:** lấy người đầu (điểm 3) chặn mất bốn người điểm 1 (tổng 4). Quyết định "lấy hay không" ảnh hưởng tương lai ⇒ cần DP. **Sửa:** DP `f[i]`.

### Lỗi 4 — Trả `f[n−1]` thay vì `max(f)`

In [23]:
def wrong_last(scores, ages):
    players = sorted(zip(ages, scores))
    f = [0] * len(players)
    for i, (_, s) in enumerate(players):
        f[i] = s + max((f[j] for j in range(i) if players[j][1] <= s), default=0)
    return f[-1]                                                  # SAI

cx = ([10, 1], [1, 2])          # người lớn tuổi hơn có điểm 1 -> đội tốt nhất là {người 0}
print("sai:", wrong_last(*cx), "| đúng:", best_brute(*cx))

sai: 1 | đúng: 10


**Vì sao sai:** đội tốt nhất không nhất thiết chứa người **cuối cùng** trong thứ tự sắp. Ở đây người cuối `(tuổi 2, điểm 1)` chỉ có `f = 1`, còn đội tốt nhất là `{(tuổi 1, điểm 10)}`. Lưu ý: với ví dụ 3 của đề, `f[-1]` tình cờ đúng — test chỉ bằng ví dụ đề sẽ không bắt được lỗi này. **Sửa:** `max(f)`.

### Lỗi 5 — Coi người cùng tuổi cũng xung đột (dùng `≤` cho tuổi khi kiểm tra cặp)

In [24]:
def conflict_wrong(i, j, scores, ages):
    return (ages[i] <= ages[j] and scores[i] > scores[j]) or (ages[j] <= ages[i] and scores[j] > scores[i])  # SAI

def wrong_same_age(scores, ages):
    n = len(scores); best = 0
    for mask in range(1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        if all(not conflict_wrong(i, j, scores, ages) for i, j in combinations(idx, 2)):
            best = max(best, sum(scores[i] for i in idx))
    return best

cx = ([7, 1, 9], [5, 5, 5])
print("sai:", wrong_same_age(*cx), "| đúng:", best_brute(*cx))

sai: 9 | đúng: 17


**Vì sao sai:** đề nói rõ "A conflict does not occur between players of the same age". Ba người cùng tuổi 5 luôn đi chung được (tổng 17), bản sai chỉ cho chọn người điểm cao nhất. Cẩn thận: trên ví dụ 2 của đề bản sai vẫn ra 16 (vì hai người cùng tuổi ở đó có điểm bằng nhau) ⇒ cần phản ví dụ riêng.
**Sửa:** chỉ xung đột khi tuổi **khác nhau hẳn** (`<`) và điểm người trẻ **lớn hơn hẳn**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ràng buộc "**mọi cặp** thỏa quan hệ thứ tự theo hai thuộc tính" ⇒ **sắp theo một thuộc tính, LIS/LNDS theo thuộc tính kia** (giống Russian Doll Envelopes, Box Stacking).
- Tối ưu **tổng** chứ không phải **độ dài** ⇒ không dùng mẹo `bisect` của LIS độ dài; dùng DP `O(n²)` hoặc BIT max.
- `n ≤ 1000` ⇒ `O(n²)` là đủ, đừng phức tạp hóa nếu chưa cần.
- Luôn kiểm: phần tử bằng nhau ở khóa 1 có được ghép không? ⇒ quyết định tie-break **cùng chiều** hay **ngược chiều** (Russian Doll dùng ngược chiều vì yêu cầu tăng nghiêm ngặt!).
- Khóa thứ hai có miền nhỏ (`≤ 1000`) ⇒ BIT trực tiếp; miền lớn ⇒ nén tọa độ.

## 15. Biến thể của bài

1. **LC 300 Longest Increasing Subsequence** — độ dài thay vì tổng.
2. **LC 354 Russian Doll Envelopes** — hai chiều **tăng nghiêm ngặt** ⇒ sắp `(w tăng, h giảm)` rồi LIS.
3. **LC 1691 Maximum Height by Stacking Cuboids** — ba chiều, sắp + DP tổng lớn nhất.
4. **LC 368 Largest Divisible Subset** — DP trên dãy đã sắp, truy vết đáp án.
5. **Trả về chính đội tốt nhất** (bài tập 5) — DP + mảng `parent`.
6. **Giới hạn số người tối đa `K`** — thêm chiều `f[i][c]`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tập con | `O(2ⁿn²)` | `O(n)` | oracle, `n ≤ 15` |
| 2. Sắp (tuổi, điểm) + DP | LNDS tổng max trên điểm | `O(n²)` | `O(n)` | lời giải chuẩn phỏng vấn |
| 3. Sắp (điểm, tuổi) + DP | LNDS tổng max trên tuổi | `O(n²)` | `O(n)` | như Cách 2, chuẩn bị cho BIT |
| 4. BIT trên tuổi | max tiền tố theo tuổi | `O(n log A)` | `O(A)` | `n` lớn, tuổi nhỏ |
| 5. BIT trên điểm nén | max tiền tố theo hạng điểm | `O(n log n)` | `O(n)` | khóa thứ hai miền lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `scores=[6,5,4,3], ages=[1,2,3,4]`; (b) `scores=[1,1,1], ages=[3,2,1]`; (c) `scores=[10,2,3,8], ages=[3,1,2,2]`.

**Bài 2 (tính tay):** `scores=[9,2,8,8,2], ages=[4,1,3,1,2]` — viết danh sách sau sắp và bảng `f`.

**Bài 3 (điền TODO):** hoàn thiện `is_nondecreasing_after_sort(team_scores, team_ages)` kiểm tra một đội có hợp lệ bằng cách sắp rồi xét các cặp **liền kề** (không xét mọi cặp).

**Bài 4 (tự viết lại):** viết lại Cách 5 (BIT + nén tọa độ) mà không xem code.

**Bài 5 (mini challenge):** viết `best_team_members(scores, ages)` trả về **danh sách chỉ số** của một đội tối ưu (truy vết bằng `parent`).

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_nondecreasing_after_sort(team_scores, team_ages):
    # TODO: sắp các cặp (tuổi, điểm), kiểm tra điểm không giảm giữa các cặp liền kề
    pass

In [26]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_bit_compressed_again(scores, ages):
    # TODO
    pass

def best_team_members(scores, ages):
    # TODO: DP như Cách 2 + parent[i]; trả list chỉ số gốc
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp theo `(tuổi, điểm)`. Mức 2: tìm dãy con không giảm tổng max. Mức 3: (a) dãy `[6,5,4,3]` giảm ⇒ `6`; (b) dãy `[1,1,1]` ⇒ `3`; (c) dãy `[2,3,8,10]` ⇒ `23`.

**Bài 2.** Mức 1: sau sắp `(1,2), (1,8), (2,2), (3,8), (4,9)`. Mức 2: `f = [2, 10, 4, 18, 27]`. Mức 3: đáp án `27` (bỏ người `(2,2)`).

**Bài 3.** Mức 1: `sorted(zip(team_ages, team_scores))`. Mức 2: tính bắc cầu của `≤` ⇒ chỉ cần xét liền kề. Mức 3: `all(p[1] <= q[1] for p, q in zip(L, L[1:]))`.

**Bài 4.** Mức 1: `rank = {v: i+1 ...}` trên `sorted(set(scores))`. Mức 2: duyệt `sorted(zip(ages, scores))`. Mức 3: `cur = query(rank[s]) + s; update(rank[s], cur)`.

**Bài 5.** Mức 1: giữ `parent[i] = j` đạt max (hoặc `-1`). Mức 2: lưu cả chỉ số gốc khi sắp: `sorted(range(n), key=lambda i: (ages[i], scores[i]))`. Mức 3: từ `argmax f` đi ngược theo `parent`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
b1 = [Solution().bestTeamScore(s, a) for s, a in [([6, 5, 4, 3], [1, 2, 3, 4]), ([1, 1, 1], [3, 2, 1]), ([10, 2, 3, 8], [3, 1, 2, 2])]]
print("Bài 1:", b1)
assert b1 == [6, 3, 23]
assert trace_dp([9, 2, 8, 8, 2], [4, 1, 3, 1, 2]) == 27

Bài 1: [6, 3, 23]
sau sắp xếp (tuổi, điểm): [(1, 2), (1, 8), (2, 2), (3, 8), (4, 9)]
i=0 (1, 2): j hợp lệ=[], max f[j]=0 -> f[0]=2
i=1 (1, 8): j hợp lệ=[0], max f[j]=2 -> f[1]=10
i=2 (2, 2): j hợp lệ=[0], max f[j]=2 -> f[2]=4
i=3 (3, 8): j hợp lệ=[0, 1, 2], max f[j]=10 -> f[3]=18
i=4 (4, 9): j hợp lệ=[0, 1, 2, 3], max f[j]=18 -> f[4]=27
f = [2, 10, 4, 18, 27] -> đáp án 27


In [28]:
# Bài 3
def is_nondecreasing_after_sort_sol(team_scores, team_ages):
    L = sorted(zip(team_ages, team_scores))
    return all(p[1] <= q[1] for p, q in zip(L, L[1:]))

rnd = random.Random(3)
for _ in range(2000):
    m = rnd.randint(0, 7)
    ts = [rnd.randint(1, 6) for _ in range(m)]; ta = [rnd.randint(1, 4) for _ in range(m)]
    assert is_nondecreasing_after_sort_sol(ts, ta) == valid_team(list(range(m)), ts, ta)
print("Bài 3: OK")

Bài 3: OK


In [29]:
# Bài 4
def best_bit_compressed_sol(scores, ages):
    vals = sorted(set(scores)); rank = {v: i + 1 for i, v in enumerate(vals)}
    m = len(vals); tree = [0] * (m + 1); ans = 0
    for a, s in sorted(zip(ages, scores)):
        x, best = rank[s], 0
        while x > 0:
            best = max(best, tree[x]); x -= x & -x
        cur = best + s
        x = rank[s]
        while x <= m:
            tree[x] = max(tree[x], cur); x += x & -x
        ans = max(ans, cur)
    return ans

rnd = random.Random(4)
for _ in range(300):
    n = rnd.randint(1, 60)
    sc = [rnd.randint(1, 10**6) for _ in range(n)]; ag = [rnd.randint(1, 30) for _ in range(n)]
    assert best_bit_compressed_sol(sc, ag) == best_dp_age(sc, ag)
print("Bài 4: OK")

Bài 4: OK


In [30]:
# Bài 5
def best_team_members_sol(scores, ages):
    n = len(scores)
    order = sorted(range(n), key=lambda i: (ages[i], scores[i]))
    f = [0] * n; parent = [-1] * n
    for p in range(n):
        i = order[p]
        best_prev, par = 0, -1
        for q in range(p):
            j = order[q]
            if scores[j] <= scores[i] and f[q] > best_prev:
                best_prev, par = f[q], q
        f[p] = best_prev + scores[i]; parent[p] = par
    p = max(range(n), key=lambda t: f[t])
    team = []
    while p != -1:
        team.append(order[p]); p = parent[p]
    return team[::-1]

team = best_team_members_sol([4, 5, 6, 5], [2, 1, 2, 1])
print("Bài 5 ví dụ 2:", team)
rnd = random.Random(5)
for _ in range(400):
    n = rnd.randint(1, 10)
    sc = [rnd.randint(1, 8) for _ in range(n)]; ag = [rnd.randint(1, 4) for _ in range(n)]
    t = best_team_members_sol(sc, ag)
    assert valid_team(t, sc, ag) and sum(sc[i] for i in t) == best_brute(sc, ag)
print("Bài 5: OK")

Bài 5 ví dụ 2: [1, 3, 2]
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sắp xếp + dãy con không giảm tổng max":**

```python
players = sorted(zip(key1, key2))           # tie-break cùng chiều
f = [0] * n
for i, (_, v) in enumerate(players):
    f[i] = val_i + max((f[j] for j in range(i) if players[j][1] <= v), default=0)
return max(f)
```

**Template BIT max tiền tố:**

```python
for s, a in sorted(zip(scores, ages)):
    cur = query(a) + s      # max trên 1..a
    update(a, cur)          # tree[x] = max(tree[x], cur)
```

**Checklist:**
- [ ] Sắp theo **tuple** `(tuổi, điểm)` (có tie-break), không chỉ theo tuổi?
- [ ] So sánh `≤` (điểm bằng nhau hợp lệ)?
- [ ] Trả `max(f)`, không phải `f[-1]`?
- [ ] Không dùng tham lam?
- [ ] Đã thử: 1 người, mọi người cùng tuổi, mọi người cùng điểm, tuổi tăng – điểm giảm?